In [1]:
import pandas as pd

df = pd.read_excel("Data_Intra.xlsx",sheet_name = "Sheet1",skiprows = 2)

arr = []
for i in df.columns:
    if "Unnamed" not in i:
        arr.append(i)
        

col_ = []

for i in range(len(arr)):
    col_.append('Dates' +  arr[i])
    col_.append(arr[i])
    col_.append('Blank ' + arr[i])

df = pd.read_excel("Data_Intra.xlsx",sheet_name = "Sheet1",skiprows = 3)

df.columns = col_[:-1]

dict_curr = {}
for i in range(len(arr)):
    temp = df[['Dates'+ arr[i], arr[i]]]
    
    temp = temp.rename({'Dates'+ arr[i]: 'Date', arr[i]:'close'},axis = 1)
    temp = temp.set_index("Date").dropna()
    
    dict_curr[arr[i]]= temp

In [2]:
def find_optimal_crosses(intraday_data, rolling_window=30):
    """
    Identify suitable crosses based on:
    - Stable intraday correlations
    - Mean-reverting pairs
    - Volatility ratios
    """
    closes = pd.DataFrame({pair: df['close'] for pair, df in intraday_data.items()}).dropna()
    returns = closes.pct_change().dropna()
    
    # Rolling correlation analysis
    corr_matrix = returns.rolling(rolling_window).corr().dropna()
    avg_corr = corr_matrix.groupby(level=1).mean()
    
    # Find pairs with historically stable correlations
    corr_stability = corr_matrix.groupby(level=0).std().mean()
    stable_pairs = corr_stability[corr_stability < 0.2].index.tolist()
    
    # Calculate half-life of mean reversion for each pair
    half_lives = {}
    for i, pair1 in enumerate(stable_pairs):
        for pair2 in stable_pairs[i+1:]:
            spread = returns[pair1] - returns[pair2]
            lag = spread.shift(1).dropna()
            delta = spread.diff().dropna()
            
            try:
                model = LinearRegression().fit(lag.values.reshape(-1,1), delta)
                hl = max(-np.log(2)/model.coef_[0], 0)
                half_lives[f"{pair1}/{pair2}"] = hl
            except:
                continue
    
    # Filter for pairs with suitable mean-reversion speed (5-50 periods)
    suitable_pairs = {k:v for k,v in half_lives.items() if 5 <= v <= 50}
    
    return {
        'correlation_matrix': avg_corr,
        'stable_pairs': stable_pairs,
        'half_lives': half_lives,
        'suitable_crosses': suitable_pairs
    }

In [3]:
find_optimal_crosses(dict_curr)

{'correlation_matrix':                  EURUSD Curncy  USDSEK Curncy  USDCHF Curncy  USDJPY Curncy  \
 EURUSD Curncy         1.000000      -0.694153      -0.635001      -0.388927   
 USDCAD Curncy        -0.496293       0.469539       0.229850       0.033703   
 USDCHF Curncy        -0.635001       0.430742       1.000000       0.585242   
 USDJPY Curncy        -0.388927       0.213047       0.585242       1.000000   
 USDNOK Curncy        -0.635724       0.693335       0.334776       0.141802   
 USDNOK Curncy.1      -0.635724       0.693335       0.334776       0.141802   
 USDSEK Curncy        -0.694153       1.000000       0.430742       0.213047   
 
                  USDCAD Curncy  USDNOK Curncy  USDNOK Curncy.1  
 EURUSD Curncy        -0.496293      -0.635724        -0.635724  
 USDCAD Curncy         1.000000       0.510106         0.510106  
 USDCHF Curncy         0.229850       0.334776         0.334776  
 USDJPY Curncy         0.033703       0.141802         0.141802  
 USDNO

## Time of day pattern identification

In [6]:
def identify_time_patterns(intraday_data, pair):
    """
    Identify optimal trading times for a pair based on:
    - Hourly return distributions
    - Volatility patterns
    - Session overlaps
    """
    df = intraday_data[pair].copy()
    df['hour'] = df.index.hour
    df['returns'] = df['close'].pct_change()
    
    # Hourly performance metrics
    hourly_stats = df.groupby('hour')['returns'].agg(['mean', 'std', 'count'])
    hourly_stats['sharpe'] = hourly_stats['mean'] / hourly_stats['std']
    
    # Identify key trading windows
    london_open = 8
    ny_open = 13
    london_ny_overlap = hourly_stats.loc[london_open:ny_open]
    
    return {
        'hourly_stats': hourly_stats,
        'optimal_hours': london_ny_overlap['sharpe'].idxmax(),
        'volatility_peaks': hourly_stats['std'].idxmax()
    }

In [8]:
for pair in dict_curr.keys():

    display(identify_time_patterns(dict_curr, pair))

{'hourly_stats':               mean       std  count    sharpe
 hour                                         
 0    -3.479187e-05  0.000593    635 -0.058647
 1    -1.118850e-06  0.000570    637 -0.001964
 2     5.785263e-06  0.000546    754  0.010591
 3    -2.363845e-05  0.000526    770 -0.044956
 4    -7.433901e-06  0.000463    771 -0.016049
 5     1.629105e-06  0.000322    744  0.005058
 6     4.341825e-05  0.000296    636  0.146576
 7     2.751990e-07  0.000386    636  0.000713
 8     5.266214e-06  0.000459    636  0.011483
 9     9.580795e-06  0.000421    636  0.022770
 10   -2.180606e-06  0.000301    636 -0.007251
 11    1.872594e-06  0.000276    636  0.006783
 12   -2.888910e-07  0.000254    636 -0.001139
 13   -1.013982e-05  0.000304    636 -0.033304
 14    2.455388e-05  0.000370    636  0.066391
 15    6.866555e-06  0.000531    636  0.012930
 16    4.270144e-05  0.000601    636  0.071010
 17   -1.711188e-05  0.000555    632 -0.030805
 18    1.848567e-05  0.000558    630  0.0331

{'hourly_stats':           mean       std  count    sharpe
 hour                                     
 0     0.000021  0.000765    635  0.027821
 1    -0.000010  0.000748    636 -0.013001
 2    -0.000039  0.000690    650 -0.056242
 3     0.000033  0.000636    701  0.052597
 4     0.000029  0.000868    755  0.033288
 5    -0.000004  0.000434    739 -0.009342
 6    -0.000038  0.000423    637 -0.089171
 7     0.000005  0.000448    636  0.010789
 8     0.000009  0.000629    636  0.014639
 9    -0.000024  0.000569    636 -0.041881
 10   -0.000004  0.000381    636 -0.009707
 11    0.000009  0.000355    636  0.025630
 12    0.000005  0.000325    636  0.015398
 13    0.000007  0.000418    636  0.017538
 14   -0.000038  0.000553    636 -0.067844
 15    0.000001  0.000790    636  0.001431
 16   -0.000053  0.000821    631 -0.064807
 17    0.000035  0.000706    630  0.049280
 18   -0.000003  0.000734    630 -0.003512
 19   -0.000033  0.000673    630 -0.048399
 20   -0.000023  0.000683    630 -0.03

{'hourly_stats':           mean       std  count    sharpe
 hour                                     
 0     0.000236  0.000905     54  0.261025
 1     0.000038  0.001691     54  0.022268
 2     0.000033  0.001206     54  0.027619
 3     0.000027  0.000915     60  0.029311
 4    -0.000075  0.000786     60 -0.095890
 5    -0.000092  0.000725     48 -0.127497
 6    -0.000267  0.000606     48 -0.440200
 7    -0.000086  0.000550     48 -0.155882
 8    -0.000245  0.001133     48 -0.216166
 9     0.000083  0.000678     53  0.122760
 10    0.000049  0.000478     54  0.102645
 11   -0.000252  0.000474     54 -0.531884
 12   -0.000077  0.000854     54 -0.090767
 13    0.000041  0.000883     54  0.046882
 14   -0.000022  0.000902     54 -0.024408
 15   -0.000185  0.001313     54 -0.141000
 16   -0.000147  0.001056     54 -0.139094
 17   -0.000131  0.000818     54 -0.160090
 18   -0.000104  0.001226     54 -0.084740
 19   -0.000276  0.000926     54 -0.298413
 20   -0.000151  0.001159     54 -0.13

{'hourly_stats':           mean       std  count    sharpe
 hour                                     
 0     0.000258  0.000832     54  0.310072
 1     0.000273  0.001348     54  0.202791
 2     0.000025  0.001150     54  0.021579
 3    -0.000116  0.000956     60 -0.121105
 4    -0.000136  0.000742     60 -0.183348
 5    -0.000142  0.000590     48 -0.241287
 6    -0.000252  0.000920     48 -0.273946
 7    -0.000079  0.000691     48 -0.114730
 8    -0.000211  0.001014     48 -0.208093
 9     0.000150  0.000921     53  0.162381
 10    0.000079  0.000551     54  0.144096
 11   -0.000287  0.000569     54 -0.503791
 12   -0.000129  0.000767     54 -0.168097
 13    0.000132  0.000829     54  0.159668
 14   -0.000026  0.000938     54 -0.028229
 15   -0.000156  0.001074     54 -0.145331
 16   -0.000040  0.000879     54 -0.044978
 17   -0.000075  0.000608     54 -0.124145
 18   -0.000186  0.000955     54 -0.194545
 19   -0.000235  0.000984     54 -0.239230
 20   -0.000024  0.001121     54 -0.02

{'hourly_stats':               mean       std  count    sharpe
 hour                                         
 0     1.374708e-05  0.000593     54  0.023171
 1    -1.372119e-04  0.000778     54 -0.176380
 2     1.013659e-04  0.000583     54  0.173920
 3     6.672395e-05  0.000631     60  0.105705
 4     5.633605e-06  0.000989     60  0.005694
 5    -1.209279e-04  0.000800     48 -0.151073
 6    -2.232988e-05  0.000589     48 -0.037906
 7    -8.302481e-05  0.000407     48 -0.204020
 8    -7.544288e-05  0.000637     48 -0.118453
 9    -1.010465e-04  0.000427     53 -0.236885
 10    2.382933e-05  0.000356     54  0.066904
 11    6.235433e-06  0.000355     54  0.017583
 12   -3.427974e-05  0.000360     54 -0.095155
 13   -2.526666e-06  0.000481     54 -0.005249
 14   -2.969292e-05  0.000580     54 -0.051216
 15   -1.834667e-05  0.000621     54 -0.029560
 16   -8.500017e-05  0.000632     54 -0.134538
 17    5.640121e-05  0.000451     54  0.125118
 18   -7.056133e-06  0.000531     54 -0.0132

{'hourly_stats':           mean       std  count    sharpe
 hour                                     
 0     0.000163  0.001244     54  0.130692
 1    -0.000443  0.001718     54 -0.257762
 2    -0.000178  0.001301     54 -0.136924
 3     0.000093  0.001154     55  0.080460
 4     0.000231  0.001538     60  0.150379
 5    -0.000117  0.000887     48 -0.131646
 6    -0.000035  0.000721     48 -0.047944
 7    -0.000060  0.000689     48 -0.086658
 8    -0.000084  0.001352     48 -0.061884
 9    -0.000144  0.000879     53 -0.163987
 10    0.000080  0.000724     54  0.111103
 11   -0.000058  0.000948     54 -0.061491
 12   -0.000111  0.000669     54 -0.166648
 13    0.000012  0.000933     54  0.012747
 14   -0.000069  0.001291     54 -0.053061
 15    0.000665  0.001895     54  0.350646
 16   -0.000283  0.001297     54 -0.217908
 17    0.000068  0.001080     54  0.063170
 18    0.000051  0.001651     54  0.031009
 19    0.000165  0.001351     54  0.122426
 20   -0.000373  0.001037     54 -0.35

{'hourly_stats':           mean       std  count    sharpe
 hour                                     
 0     0.000163  0.001244     54  0.130692
 1    -0.000443  0.001718     54 -0.257762
 2    -0.000178  0.001301     54 -0.136924
 3     0.000093  0.001154     55  0.080460
 4     0.000231  0.001538     60  0.150379
 5    -0.000117  0.000887     48 -0.131646
 6    -0.000035  0.000721     48 -0.047944
 7    -0.000060  0.000689     48 -0.086658
 8    -0.000084  0.001352     48 -0.061884
 9    -0.000144  0.000879     53 -0.163987
 10    0.000080  0.000724     54  0.111103
 11   -0.000058  0.000948     54 -0.061491
 12   -0.000111  0.000669     54 -0.166648
 13    0.000012  0.000933     54  0.012747
 14   -0.000069  0.001291     54 -0.053061
 15    0.000665  0.001895     54  0.350646
 16   -0.000283  0.001297     54 -0.217908
 17    0.000068  0.001080     54  0.063170
 18    0.000051  0.001651     54  0.031009
 19    0.000165  0.001351     54  0.122426
 20   -0.000373  0.001037     54 -0.35

Goal:
Find repeatable intraday behaviors — e.g., mean reversion, breakout bias, volatility spikes — around specific times (session opens, closes, fixings, etc.)

4. Extract Patterns & Insights
Look for volatility spikes around:
07:00 UTC (London open)
12:30–13:30 UTC (US economic releases)
15:00–16:00 UTC (London fix / NY open overlap)
Find mean-reverting windows (e.g., 00:00–03:00 UTC during Asia)
Spot consistent directional bias in early London or NY

In [13]:
# 📦 Imports
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# 🗓️ Download 5-minute EUR/USD data from Yahoo
symbol = "EURUSD=X"
df = yf.download(symbol, interval="5m", start="2025-04-01", end="2025-04-13")

# 🕰️ Convert index to UTC (already tz-aware)
df.index = df.index.tz_convert('UTC')

display(df)
# 📈 Add return and time-of-day features
df['return'] = df['Close'].pct_change()
df['hour_min'] = df.index.strftime('%H:%M')

display(df)
# 📊 Group by each 5-minute time window across all days
tod_stats = df.groupby('hour_min').agg({
    'return': ['mean', 'std'],
    'Volume': 'mean'
}).dropna()

# 🧼 Clean up column names
tod_stats.columns = ['mean_return', 'volatility', 'avg_volume']

# 📉 Plot: Mean Return
plt.figure(figsize=(14, 4))
plt.plot(tod_stats.index, tod_stats['mean_return'])
plt.title('📈 Average Return by Time of Day (UTC) - EUR/USD')
plt.xticks(rotation=90)
plt.grid(True)
plt.tight_layout()
plt.show()

# ⚡ Plot: Volatility
plt.figure(figsize=(14, 4))
plt.plot(tod_stats.index, tod_stats['volatility'], color='orange')
plt.title('⚡ Volatility by Time of Day (UTC) - EUR/USD')
plt.xticks(rotation=90)
plt.grid(True)
plt.tight_layout()
plt.show()

# 🔊 Plot: Volume (activity proxy)
plt.figure(figsize=(14, 4))
plt.plot(tod_stats.index, tod_stats['avg_volume'], color='green')
plt.title('🔊 Average Volume by Time of Day (UTC) - EUR/USD')
plt.xticks(rotation=90)
plt.grid(True)
plt.tight_layout()
plt.show()

# 🧠 Optional: Display early preview of stats
print(tod_stats.head())


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X
Datetime,,,,,
2025-03-31 23:00:00+00:00,1.082134,1.082134,1.082134,1.082134,0
2025-03-31 23:05:00+00:00,1.082251,1.082251,1.082134,1.082134,0
2025-03-31 23:10:00+00:00,1.082134,1.082134,1.082134,1.082134,0
2025-03-31 23:15:00+00:00,1.082017,1.082134,1.082017,1.082134,0
2025-03-31 23:20:00+00:00,1.081783,1.081900,1.081783,1.081783,0
...,...,...,...,...,...
2025-04-11 21:05:00+00:00,1.136364,1.136364,1.136364,1.136364,0
2025-04-11 21:10:00+00:00,1.136364,1.136364,1.136364,1.136364,0


Price,Close,High,Low,Open,Volume,return,hour_min
Ticker,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,,
Datetime,,,,,,,
2025-03-31 23:00:00+00:00,1.082134,1.082134,1.082134,1.082134,0,NaN,23:00
2025-03-31 23:05:00+00:00,1.082251,1.082251,1.082134,1.082134,0,0.000108,23:05
2025-03-31 23:10:00+00:00,1.082134,1.082134,1.082134,1.082134,0,-0.000108,23:10
2025-03-31 23:15:00+00:00,1.082017,1.082134,1.082017,1.082134,0,-0.000108,23:15
2025-03-31 23:20:00+00:00,1.081783,1.081900,1.081783,1.081783,0,-0.000216,23:20
...,...,...,...,...,...,...,...
2025-04-11 21:05:00+00:00,1.136364,1.136364,1.136364,1.136364,0,0.000000,21:05
2025-04-11 21:10:00+00:00,1.136364,1.136364,1.136364,1.136364,0,0.000000,21:10


KeyError: "Column(s) ['Volume', 'return'] do not exist"